# Practical 5 — Sales Data Clustering using K-Means
### Beginner-friendly ML practical and viva guide

**Dataset:** `sales_data_sample.csv`  
**Aim:** Group similar sales records into clusters and use the **Elbow Method** to choose a suitable number of clusters.

**Run instructions:** Keep `sales_data_sample.csv` in the same folder as this notebook. Run cells from top to bottom using **Shift + Enter**.

**Note:** This notebook focuses on your simpler K-Means code. Your uploaded original assignment also mentions hierarchical clustering, so a small optional section is included at the end.

## 1. What is clustering?
Imagine a shop with thousands of orders. We want to group similar orders together without already knowing their group names.

- **Clustering:** Putting similar records in the same group.
- **Unsupervised learning:** The dataset has no correct cluster label to predict.
- **K-Means:** Starts with `k` centers, assigns each record to its nearest center, then updates the centers repeatedly.
- **Centroid:** The average position of records in a cluster.
- **k:** Number of clusters. Example: `k=3` means three groups.

**Memory trick:** K = how many groups; Means = average centers.

## 2. Import libraries
Libraries are ready-made toolboxes.

- `pandas` reads and manages tables.
- `matplotlib` draws graphs.
- `StandardScaler` puts numerical columns on comparable scales.
- `KMeans` creates clusters.

In [ ]:
# Import required libraries
import pandas as pd                       # Work with datasets
import matplotlib.pyplot as plt          # Draw graphs
from sklearn.preprocessing import StandardScaler  # Scale numbers
from sklearn.cluster import KMeans        # K-Means clustering

## 3. Load and inspect the dataset

- `read_csv()` opens the CSV file.
- `encoding="latin1"` helps read special characters in this dataset.
- `head()` displays the first five rows.
- `shape` gives (number of rows, number of columns).

In [ ]:
# Load the sales data file (keep CSV beside notebook)
df = pd.read_csv("sales_data_sample.csv", encoding="latin1")

print("Dataset size:", df.shape)   # Rows and columns
print(df.head())                  # First 5 records

## 4. Choose features
We will group orders based on four numeric columns:

| Column | Simple meaning |
|---|---|
| `QUANTITYORDERED` | Number of items ordered |
| `PRICEEACH` | Price per item |
| `SALES` | Sales value of the order line |
| `MSRP` | Manufacturer's suggested retail price |

`dropna()` removes rows missing any of these values. **There is no `y` target column** because clustering is unsupervised.

In [ ]:
# Choose four numeric columns for clustering
X = df[["QUANTITYORDERED", "PRICEEACH", "SALES", "MSRP"]].dropna()

print("Records used:", X.shape)  # How many rows and features

## 5. Scale the data (important)
Why scale? Imagine `SALES` is 3000 while `QUANTITYORDERED` is 30. Without scaling, the larger-number column can dominate distance calculations.

`StandardScaler()` changes each feature to have approximately **mean 0 and standard deviation 1**.

- `fit_transform(X)` learns the mean/spread and applies scaling.
- K-Means uses distances, so scaling matters.

In [ ]:
# Scale features so large numbers don't dominate distance
scaler = StandardScaler()         # Create scaler
X = scaler.fit_transform(X)       # Scale all four columns

print("Scaled data shape:", X.shape)

## 6. Elbow Method — find a good k
We try **k = 1 to 10** clusters. For each k, K-Means computes **inertia**, also called WCSS (within-cluster sum of squares).

- **Lower WCSS** means records are closer to their assigned cluster centers.
- WCSS almost always falls when k increases, so simply choosing the smallest WCSS is **not** correct.
- Look for the **bend (elbow)** where improvement starts slowing.

**Loop explanation:** `for k in range(1, 11)` repeats the code 10 times: k = 1, 2, ..., 10.

`wcss.append(...)` saves each result in a list.

In [ ]:
# Test 1 to 10 clusters
wcss = []                                # Empty list for WCSS values

for k in range(1, 11):                 # k = 1, 2, ..., 10
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    model.fit(X)                       # Make k clusters
    wcss.append(model.inertia_)        # Save WCSS for this k

# Draw the elbow graph
plt.plot(range(1, 11), wcss, marker="o")
plt.xlabel("Number of clusters (k)")
plt.ylabel("WCSS / Inertia")
plt.title("Elbow Method")
plt.xticks(range(1, 11))
plt.show()

## 7. Choose k and create final clusters
Look at the graph above. The example uses **k = 3**, which is also the choice described in your uploaded lab notes. You can change it if your elbow plot suggests a different value.

- `fit_predict(X)` trains K-Means **and** gives each record a cluster label.
- Labels 0, 1, 2 are just group names; they do **not** mean best, middle, worst.
- `value_counts()` counts records in each group.

In [ ]:
# Final K-Means model
k = 3                                      # Choose 3 groups (change if needed)
model = KMeans(n_clusters=k, random_state=42, n_init=10)
labels = model.fit_predict(X)              # Assign each record to a cluster

print("Records in each cluster:")
print(pd.Series(labels).value_counts().sort_index())

## 8. Visualize clusters (simple graph)
For an easy 2D graph, use two of the **scaled** columns: quantity and price. Colors show the clusters computed using **all four** features.

**Important:** A two-column picture cannot fully show the four-dimensional clustering. Overlap on the graph is possible.

In [ ]:
# Plot clusters using first two scaled columns
plt.scatter(X[:, 0], X[:, 1], c=labels, cmap="viridis", s=12)
plt.xlabel("Quantity Ordered (scaled)")
plt.ylabel("Price Each (scaled)")
plt.title("K-Means Sales Clusters")
plt.show()

## 9. What does the output mean?
- **Elbow plot:** Helps choose a reasonable number of groups.
- **Cluster counts:** Show how many orders are in each group.
- **Scatter plot:** Shows different groups in different colors.

**Don't say** “Cluster 2 is most profitable” just because it has label 2. You would need to inspect sales and profit data before making a business claim.

**No accuracy score or confusion matrix here:** There are no known correct cluster labels, so this is not a classification practical.

## 10. Key viva questions and answers

1. **What is clustering?** Grouping similar records.
2. **Is K-Means supervised?** No, it is unsupervised learning.
3. **What does K mean?** Number of clusters.
4. **What is a centroid?** Average center of the records in one cluster.
5. **How does K-Means work?** Assign points to nearest centers, update centers, repeat.
6. **What is the Elbow Method?** A graph-based method to choose k where WCSS improvement slows.
7. **What is WCSS?** Sum of squared distances from points to their cluster centers.
8. **Why use StandardScaler?** To prevent features with large numerical scales from dominating distances.
9. **What is `fit()`?** It learns cluster centers from data.
10. **What is `fit_predict()`?** It learns clusters and returns cluster labels.
11. **What is `n_init=10`?** Try 10 initial centroid configurations and keep the best result.
12. **What is `random_state=42`?** Makes randomized initialization reproducible.
13. **What does `range(1,11)` do?** Generates 1 through 10.
14. **What does `append()` do?** Adds a value to the end of a list.
15. **Why no train-test split?** This is exploratory unsupervised clustering, not prediction of known target labels.
16. **Why no accuracy?** No true class labels exist for comparing predictions.
17. **Are labels 0, 1, 2 ranked?** No, their numbers are arbitrary.
18. **What is PCA?** A technique that compresses many numeric features into fewer dimensions, often for plotting.
19. **K-Means vs KNN?** K-Means groups unlabeled data; KNN predicts known classes from labeled neighbors.
20. **What is hierarchical clustering?** It builds a tree-like hierarchy of groups, often shown as a dendrogram.

## 11. Last-minute memory trick
**Import → Load → Select 4 columns → Scale → Elbow loop → Choose k → Fit/Predict → Count → Plot**

Most important lines to remember:

```python
X = df[["QUANTITYORDERED", "PRICEEACH", "SALES", "MSRP"]].dropna()
X = StandardScaler().fit_transform(X)
model = KMeans(n_clusters=3, random_state=42, n_init=10)
labels = model.fit_predict(X)
```

If the teacher asks about the loop, explain that you test different k values and save `model.inertia_` to make the elbow graph.

## 12. OPTIONAL — Hierarchical clustering
Your original assignment objective also mentions hierarchical clustering, although the shorter code you shared covers K-Means only. If your examiner asks for hierarchical clustering, run the following extra cell.

- **Agglomerative clustering:** Starts with individual records and merges similar groups.
- Unlike K-Means, it does not repeatedly update mean centers.
- The labels are still arbitrary group IDs.

In [ ]:
# OPTIONAL: Only if your teacher asks for hierarchical clustering
from sklearn.cluster import AgglomerativeClustering

# Use a smaller sample to keep this optional example quick
X_small = X[:500]
hierarchical = AgglomerativeClustering(n_clusters=3)
h_labels = hierarchical.fit_predict(X_small)

print("Hierarchical cluster counts:")
print(pd.Series(h_labels).value_counts().sort_index())

## 13. Common errors

- **FileNotFoundError:** Put `sales_data_sample.csv` beside the notebook and open that folder in VS Code.
- **ModuleNotFoundError:** Install packages using `%pip install pandas matplotlib scikit-learn` in a separate notebook cell.
- **NameError: X is not defined:** Run the earlier cells first.
- **Different cluster numbers:** Cluster IDs are arbitrary; `random_state` helps reproducibility.

**Dataset note:** `sales_data_sample.csv` was referenced in your notes and code, but it was not attached with this message. The notebook needs that CSV file to run.